# Mamba: State Space Model - 실습 코드 2: SSM (State Space Model) 직접 구현

- Tutorial ID: `expand-mamba-ssm`
- Tutorial: Mamba: State Space Model
- Section ID: `expand-mamba-ssm-code-2`
- Section: 실습 코드 2: SSM (State Space Model) 직접 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: SSM (State Space Model) 직접 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ── S4 스타일 State Space Model 기본 구현 ──
class SimpleSSM(nn.Module):
    """Selective State Space Model (Mamba의 핵심) 간소화 구현
    
    연속 시간: h'(t) = Ah(t) + Bx(t), y(t) = Ch(t) + Dx(t)
    이산화:    h_k = Ā·h_{k-1} + B̄·x_k,  y_k = C·h_k + D·x_k
    """
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = int(d_model * expand)
        
        # 입력 프로젝션
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        
        # 1D 컨볼루션 (로컬 컨텍스트)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner, bias=True
        )
        
        # SSM 파라미터 (Selective = 입력 의존적)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        
        # dt (step size) 프로젝션
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)
        
        # A 파라미터 (대각 행렬)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(self.d_inner, 1)))
        
        # D (skip connection)
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        # 출력 프로젝션
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
        def forward(self, x):
        """x: (batch, seq_len, d_model)"""
        batch, seq_len, _ = x.shape
        
        # 입력 프로젝션 + 게이트
        xz = self.in_proj(x)  # (batch, seq, d_inner * 2)
        x_branch, z = xz.chunk(2, dim=-1)  # 각 (batch, seq, d_inner)
        
        # 1D 컨볼루션 (로컬 컨텍스트 포착)
        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x_act = F.silu(x_conv)
        
        # SSM 처리 (선택적)
        y = self._ssm_scan(x_act)
        
        # 게이트 적용
        y = y * F.silu(z)
        
        # 출력 프로젝션
        return self.out_proj(y)
    
        def _ssm_scan(self, x):
        """SSM 순차 스캔 (recurrent mode)"""
        batch, seq_len, d_inner = x.shape
        
        # Selective 파라미터 계산 (입력 의존적!)
        x_dbl = self.x_proj(x)  # (batch, seq, d_state*2 + 1)
                B = x_dbl[:, :, :self.d_state]               # (batch, seq, d_state)
        C = x_dbl[:, :, self.d_state:self.d_state*2]  # (batch, seq, d_state)
        dt = F.softplus(self.dt_proj(x_dbl[:, :, -1:]))  # (batch, seq, d_inner)
        
        # A를 이산화
                A = -torch.exp(self.A_log)  # (d_inner, d_state) 음수화
        
        # 순차 스캔
        h = torch.zeros(batch, d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        
                for t in range(seq_len):
            # h_k = Ā·h_{k-1} + B̄·x_k (이산화 via ZOH)
            dt_t = dt[:, t, :].unsqueeze(-1)  # (batch, d_inner, 1)
            dA = torch.exp(A * dt_t)           # (batch, d_inner, d_state)
            dB = B[:, t, :].unsqueeze(1) * dt_t  # (batch, d_inner, d_state)
            
            h = h * dA + dB * x[:, t, :].unsqueeze(-1)  # 상태 갱신
            
            # y_k = C·h_k + D·x_k
            y_t = torch.einsum('bnd,bd->bn', h, C[:, t, :].unsqueeze(1).expand(-1, d_inner, -1).mean(dim=-1).unsqueeze(1).expand(-1, d_inner))
            # 단순화된 출력
            y_t = (h * C[:, t, :].unsqueeze(1)).sum(-1) + self.D * x[:, t, :]
            ys.append(y_t)
        
        return torch.stack(ys, dim=1)  # (batch, seq, d_inner)


# ── Mamba 스타일 블록 ──
class MambaBlock(nn.Module):
    """Mamba 블록: SSM + Residual + Norm"""
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ssm = SimpleSSM(d_model, d_state, d_conv, expand)
    
        def forward(self, x):
        return x + self.ssm(self.norm(x))


# ── 테스트 ──
print("=== SSM (Mamba 스타일) 테스트 ===\n")

# 단일 블록 테스트
block = MambaBlock(d_model=256, d_state=16, d_conv=4, expand=2)
x = torch.randn(2, 64, 256)
out = block(x)
params = sum(p.numel() for p in block.parameters())
print(f"단일 블록: {x.shape} → {out.shape}, 파라미터: {params:,}")

# 작은 Mamba 모델
class TinyMambaLM(nn.Module):
        def __init__(self, vocab_size=32000, d_model=256, n_layers=4, d_state=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(d_model, d_state) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
        def forward(self, idx):
                x = self.embed(idx)
                for block in self.blocks:
                        x = block(x)
        return self.head(self.norm(x))

model = TinyMambaLM(vocab_size=32000, d_model=256, n_layers=4)
params = sum(p.numel() for p in model.parameters())
x = torch.randint(0, 32000, (2, 128))
out = model(x)
print(f"TinyMambaLM: {x.shape} → {out.shape}, 파라미터: {params:,}")
print(f"\n→ O(N) 시간 복잡도로 시퀀스 처리 (Attention의 O(N²) 대비!)")